## Forth Question

In [0]:
from pyspark.sql.functions import *

##### First Task
###### 1. Read JSON file provided in the attachment using the dynamic function  

In [0]:
df = spark.read.option("multiline", "true").json("file:/Workspace/Users/jaygediya1802@gmail.com/pysparkAssignment/nested_json_file.json")
df.printSchema()

##### Second Task
###### 2. flatten the data frame which is a custom schema

In [0]:
flatten_df = df.withColumn("employee", explode("employees"))\
      .select(
          col("id"),
          col("properties.name").alias("name"),
          col("properties.storeSize").alias("storeSize"),
          col("employee.empId").alias("empId"),
          col("employee.empName").alias("empName")
      )
display(flatten_df)

##### Third Task
###### 3. find out the record count when flattened and when it's not flattened(find out the difference why you are getting more count)

In [0]:
print("Before the Flatten : ", df.count())

print("After the Flatten : ", flatten_df.count())

##### Fourth Task
###### 4. Differentiate the difference using explode, explode outer, posexplode functions

In [0]:
df.select(explode("employees")).show(truncate=False)

In [0]:
df.select(explode_outer("employees")).show(truncate=False)

In [0]:
df.select(posexplode("employees")).show(truncate=False)

##### Fifth Task
###### 5. Filter the id which is equal to 0001  

In [0]:
display(df.filter(col("id") == 1001))

##### Sixth Task
###### 6. convert the column names from camel case to snake case 

In [0]:
flatten_df = df.withColumn("employee", explode("employees"))\
      .select(
          col("id"),
          col("properties.name").alias("name"),
          col("properties.storeSize").alias("store_size"),
          col("employee.empId").alias("emp_id"),
          col("employee.empName").alias("emp_name")
      )
display(flatten_df)

##### Seventh Task
###### 7. Add a new column named load_date with the current date

In [0]:
flatten_df = flatten_df.withColumn(
    "load_date",
    current_date()
)
display(flatten_df)

##### Eighth Task
###### 8. create 3 new columns as year, month, and day from the load_date column 

In [0]:
flatten_df = (
    flatten_df
    .withColumn("year", year("load_date"))
    .withColumn("month", month("load_date"))
    .withColumn("day", dayofmonth("load_date"))
)
display(flatten_df)

##### Ninth Task
###### 9. write data frame to a table with the Database name as employee and table name as employee_details with overwrite mode, format as JSON and partition based on (year, month, day) using replacing where condition on year, month, day 

In [0]:
partitions = flatten_df.select("year", "month", "day").distinct().collect()
replace_condition = " OR ".join(
    [
        f"(year = {r['year']} AND month = {r['month']} AND day = {r['day']})"
        for r in partitions
    ]
)

flatten_df.write\
        .mode("overwrite")\
        .format("delta")\
        .option("replaceWhere", replace_condition)\
        .partitionBy("year", "month", "day")\
        .saveAsTable("employee_details")